In [14]:
from __future__ import annotations

import asyncio
import json
import logging
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import AsyncIterator

import aiohttp
from bs4 import BeautifulSoup
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

logger = logging.getLogger(__name__)

HEADERS = {"User-Agent": "Mozilla/5.0 (research-scraper/1.0; contact: your@email.com)"}
CONCURRENCY = 12
REQUEST_TIMEOUT = 30
RETRY_ATTEMPTS = 4
BASE_URL = "https://www.rba.gov.au"


@dataclass
class Document:
    url: str
    source_type: str
    year: int | None
    title: str
    text: str
    metadata: dict = field(default_factory=dict)


def _abs(href: str) -> str:
    """Converts a relative RBA href to an absolute URL.

    Args:
        href: The href attribute value from an anchor tag.

    Returns:
        Absolute URL string.
    """
    if href.startswith("http"):
        return href
    return BASE_URL + (href if href.startswith("/") else f"/{href}")


def _build_index_jobs(
    years: list[int],
    include_speeches: bool,
    include_smp: bool,
    include_fsr: bool,
) -> list[tuple[str, str, int]]:
    """Builds the full list of index page jobs across all source types.

    Args:
        years: List of years to scrape.
        include_speeches: Whether to include governor speeches.
        include_smp: Whether to include Statements on Monetary Policy.
        include_fsr: Whether to include Financial Stability Reviews.

    Returns:
        List of (index_url, source_type, year) tuples.
    """
    jobs: list[tuple[str, str, int]] = [
        (f"{BASE_URL}/monetary-policy/rba-board-minutes/{yr}/", "minutes", yr)
        for yr in years
    ]
    if include_speeches:
        jobs += [
            (f"{BASE_URL}/speeches/{yr}/", "speech", yr)
            for yr in years
        ]
    if include_smp:
        jobs += [
            (f"{BASE_URL}/publications/smp/{yr}/{yr}{mo}/", "smp", yr)
            for yr in years
            for mo in ("02", "05", "08", "11")
        ]
    if include_fsr:
        jobs += [
            (f"{BASE_URL}/publications/fsr/{yr}/{yr}{mo}/", "fsr", yr)
            for yr in years
            for mo in ("04", "10")
        ]
    return jobs


def _find_document_links(html: str, source_type: str, year: int) -> list[str]:
    """Extracts document URLs from an RBA index page.

    Args:
        html: Raw HTML of the index page.
        source_type: One of 'minutes', 'speech', 'smp', or 'fsr'.
        year: The year being scraped, used to anchor regex patterns.

    Returns:
        Deduplicated list of absolute document URLs.
    """
    patterns = {
        "minutes": re.compile(rf"{year}-\d{{2}}-\d{{2}}\.html"),
        "speech":  re.compile(rf"speeches/{year}/"),
        "smp":     re.compile(r"publications/smp/"),
        "fsr":     re.compile(r"publications/fsr/"),
    }
    pattern = patterns.get(source_type, re.compile(r"\.html$"))
    soup = BeautifulSoup(html, "lxml")
    return list({
        _abs(a["href"])
        for a in soup.find_all("a", href=True)
        if pattern.search(a["href"])
    })


def _extract_text(html: str) -> str:
    """Extracts clean body text from an RBA document page.

    Args:
        html: Raw HTML of the document page.

    Returns:
        Newline-separated paragraph text, filtered to substantive content.
    """
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    content = (
        soup.find("div", class_="article-content")
        or soup.find("main")
        or soup.find("div", id="content")
        or soup.body
    )
    if content is None:
        return ""
    return "\n\n".join(
        p.get_text(separator=" ", strip=True)
        for p in content.find_all("p")
        if len(p.get_text(strip=True)) > 40
    )


def _extract_title(html: str) -> str:
    """Extracts the page title from an RBA document page.

    Args:
        html: Raw HTML of the document page.

    Returns:
        Title string from h1 or <title> tag, or 'untitled' if absent.
    """
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    if h1:
        return h1.get_text(strip=True)
    title = soup.find("title")
    return title.get_text(strip=True) if title else "untitled"


def _make_fetch(session: aiohttp.ClientSession, semaphore: asyncio.Semaphore):
    """Creates a semaphore-bound, retry-decorated async fetch function.

    Args:
        session: Active aiohttp client session.
        semaphore: Concurrency limiter.

    Returns:
        Async callable that fetches a URL and returns its HTML.
    """
    @retry(
        retry=retry_if_exception_type((aiohttp.ClientError, asyncio.TimeoutError)),
        stop=stop_after_attempt(RETRY_ATTEMPTS),
        wait=wait_exponential(multiplier=1, min=2, max=15),
        before_sleep=before_sleep_log(logger, logging.WARNING),
        reraise=True,
    )
    async def _fetch(url: str) -> str:
        async with semaphore:
            async with session.get(url, headers=HEADERS) as response:
                response.raise_for_status()
                return await response.text()
    return _fetch


async def _crawl_index(fetch, index_url: str, source_type: str, year: int) -> list[str]:
    """Fetches an index page and returns all discovered document URLs.

    Args:
        fetch: Retry-decorated async fetch callable.
        index_url: URL of the index page to crawl.
        source_type: Source category label.
        year: Year being crawled.

    Returns:
        List of absolute document URLs found on the index page.
    """
    try:
        html = await fetch(index_url)
        return _find_document_links(html, source_type, year)
    except Exception as exc:
        logger.warning("Index fetch failed %s: %s", index_url, exc)
        return []


async def _scrape_document(fetch, url: str, source_type: str, year: int) -> Document | None:
    """Fetches and parses a single RBA document page.

    Args:
        fetch: Retry-decorated async fetch callable.
        url: URL of the document to scrape.
        source_type: Source category label.
        year: Publication year.

    Returns:
        Parsed Document instance, or None if fetching or parsing fails.
    """
    try:
        html = await fetch(url)
        text = _extract_text(html)
        if not text:
            return None
        return Document(
            url=url,
            source_type=source_type,
            year=year,
            title=_extract_title(html),
            text=text,
        )
    except Exception as exc:
        logger.warning("Document fetch failed %s: %s", url, exc)
        return None


async def scrape_all(
    years: list[int] | None = None,
    include_speeches: bool = True,
    include_smp: bool = True,
    include_fsr: bool = True,
) -> AsyncIterator[Document]:
    """Scrapes all RBA documents for the given years, yielding results as they complete.

    Args:
        years: Years to scrape. Defaults to 2010–2025 inclusive.
        include_speeches: Whether to include governor speeches.
        include_smp: Whether to include Statements on Monetary Policy.
        include_fsr: Whether to include Financial Stability Reviews.

    Yields:
        Document instances as each page is successfully scraped.
    """
    if years is None:
        years = list(range(2010, 2026))

    semaphore = asyncio.Semaphore(CONCURRENCY)
    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
    connector = aiohttp.TCPConnector(limit=CONCURRENCY * 2, keepalive_timeout=60)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        fetch = _make_fetch(session, semaphore)
        index_jobs = _build_index_jobs(years, include_speeches, include_smp, include_fsr)

        index_results = await asyncio.gather(*[
            _crawl_index(fetch, url, source_type, year)
            for url, source_type, year in index_jobs
        ])

        doc_tasks = [
            asyncio.create_task(_scrape_document(fetch, doc_url, source_type, year))
            for (_, source_type, year), doc_urls in zip(index_jobs, index_results)
            for doc_url in doc_urls
        ]

        logger.info("Scraping %d documents", len(doc_tasks))

        for coro in asyncio.as_completed(doc_tasks):
            doc = await coro
            if doc is not None:
                yield doc


def run_and_save(
    output_path: str | Path = "rba_corpus.jsonl",
    years: list[int] | None = None,
    **kwargs,
) -> int:
    """Scrapes all RBA documents and streams results to a JSONL file.

    Args:
        output_path: Destination file path for the JSONL corpus.
        years: Years to scrape. Defaults to 2010–2025 inclusive.
        **kwargs: Additional keyword arguments forwarded to scrape_all.

    Returns:
        Total number of documents successfully saved.
    """
    output_path = Path(output_path)
    count = 0

    async def _run():
        nonlocal count
        with output_path.open("w", encoding="utf-8") as fh:
            async for doc in scrape_all(years=years, **kwargs):
                fh.write(json.dumps({
                    "url": doc.url,
                    "source_type": doc.source_type,
                    "year": doc.year,
                    "title": doc.title,
                    "text": doc.text,
                }) + "\n")
                count += 1
                if count % 10 == 0:
                    logger.info("Saved %d documents...", count)

    asyncio.run(_run())
    return count

In [15]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
total = run_and_save(
    output_path="rba_corpus.jsonl",
    years=list(range(2010, 2026)),
    include_speeches=True,
    include_smp=True,
    include_fsr=True,
)
print(f"\nDone. {total} documents saved to rba_corpus.jsonl")

WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2010/201002/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2010/201005/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2011/201105/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2012/201205/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch in 2.0 seconds as it raised ClientResponseError: 404, message='Not Found', url='https://www.rba.gov.au/publications/smp/2011/201108/'.
WARNING Retrying __main__._make_fetch.<locals>._fetch i


Done. 216 documents saved to rba_corpus.jsonl


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from transformers import pipeline
from .autonotebook import tqdm as notebook_tqdm

class MacroSentimentBuilder:
    """Processes an RBA JSONL corpus into a macroeconomic sentiment index.

    Attributes:
        classifier: HuggingFace sentiment analysis pipeline.
    """

    _DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})")
    _MONTH_RE = re.compile(r"/(\d{4})(\d{2})/?$")
    _CHUNK_SIZE = 2000
    _MAX_CHUNKS = 5
    _ROLLING_WINDOW = 3

    def __init__(self, model_name: str = "ProsusAI/finbert", device: int = -1) -> None:
        """Initialises the sentiment pipeline.

        Args:
            model_name: HuggingFace model identifier.
            device: Device index for inference. -1 for CPU, 0+ for GPU.
        """
        self.classifier = pipeline(
            "sentiment-analysis",
            model=model_name,
            device=device,
            truncation=True,
            max_length=512,
        )

    def _extract_date(self, url: str) -> pd.Timestamp:
        """Parses a publication date from an RBA document URL.

        Args:
            url: Document URL, expected to contain a date or year-month segment.

        Returns:
            Parsed Timestamp, or NaT if no date pattern is matched.
        """
        if m := self._DATE_RE.search(url):
            return pd.to_datetime(m.group(1))
        if m := self._MONTH_RE.search(url):
            return pd.to_datetime(f"{m.group(1)}-{m.group(2)}-01")
        return pd.NaT

    def _score_text(self, text: str) -> float:
        """Computes a signed sentiment score for a document.

        Splits the text into fixed-size chunks, runs batch inference, and
        returns the mean signed score across all chunks. Positive labels
        contribute a positive score, negative labels a negative score, and
        neutral labels contribute zero.

        Args:
            text: Raw document text.

        Returns:
            Mean signed sentiment score in [-1, 1], or 0.0 for empty input.
        """
        chunks = [
            text[i: i + self._CHUNK_SIZE]
            for i in range(0, len(text), self._CHUNK_SIZE)
        ][: self._MAX_CHUNKS]

        if not chunks:
            return 0.0

        label_sign = {"positive": 1.0, "negative": -1.0, "neutral": 0.0}
        scores = [
            label_sign.get(result["label"], 0.0) * result["score"]
            for result in self.classifier(chunks)
        ]
        return float(np.mean(scores))

    def _load_records(self, path: Path) -> list[dict[str, Any]]:
        """Streams and scores all valid documents from a JSONL corpus file.

        Args:
            path: Path to the JSONL corpus file.

        Returns:
            List of dicts with 'date', 'source', and 'sentiment' keys.
        """
        records = []
        with path.open("r", encoding="utf-8") as fh:
            for line in fh:
                if not line.strip():
                    continue
                doc = json.loads(line)
                dt = self._extract_date(doc["url"])
                if pd.isna(dt):
                    continue
                records.append({
                    "date": dt,
                    "source": doc["source_type"],
                    "sentiment": self._score_text(doc["text"]),
                })
        return records

    def build_index(self, jsonl_path: str | Path) -> pd.DataFrame:
        """Builds a smoothed daily RBA sentiment index from a JSONL corpus.

        Scores each document, averages by date, and applies a rolling mean
        to produce a smoothed regime signal.

        Args:
            jsonl_path: Path to the JSONL corpus produced by the scraper.

        Returns:
            DataFrame indexed by date with 'sentiment' and 'rba_regime' columns.
        """
        records = self._load_records(Path(jsonl_path))
        df = (
            pd.DataFrame(records)
            .dropna(subset=["date"])
            .groupby("date")["sentiment"]
            .mean()
            .sort_index()
            .to_frame()
        )
        df["rba_regime"] = (
            df["sentiment"]
            .rolling(window=self._ROLLING_WINDOW, min_periods=1)
            .mean()
        )
        return df

/opt/anaconda3/envs/OptionPricer/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_sentiment = MacroSentimentBuilder().build_index("rba_corpus.jsonl")
df_sentiment.to_csv("rba_sentiment.csv")
df_sentiment

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 43215.87it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,sentiment,rba_regime
date,,
2022-02-02,-0.080184,-0.080184
2022-02-11,-0.228616,-0.154400
2022-02-22,0.000000,-0.102934
2022-03-22,-0.106254,-0.111623
2022-05-03,-0.187913,-0.098056
...,...,...
2025-11-20,0.000000,-0.049252
2025-11-26,-0.169301,-0.056434
2025-12-09,-0.076355,-0.081886


In [3]:
df_sentiment

,sentiment,rba_regime
date,,
2022-02-02,-0.080184,-0.080184
2022-02-11,-0.228616,-0.154400
2022-02-22,0.000000,-0.102934
2022-03-22,-0.106254,-0.111623
2022-05-03,-0.187913,-0.098056
...,...,...
2025-11-20,0.000000,-0.049252
2025-11-26,-0.169301,-0.056434
2025-12-09,-0.076355,-0.081886


In [4]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd

_TERM_PATTERNS: dict[str, re.Pattern] = {
    term: re.compile(rf"\b{re.escape(term)}\b", re.IGNORECASE)
    for term in [
        "LNG",
        "liquefied natural gas",
        "gas tax",
        "windfall tax",
        "PRRT",
        "petroleum resources rent tax",
        "gas export",
        "energy security",
        "Santos",
        "Chevron",
        "Shell",
    ]
}

_DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})")
_MONTH_RE = re.compile(r"/(\d{4})(\d{2})/?$")


def _extract_date(url: str) -> pd.Timestamp:
    if m := _DATE_RE.search(url):
        return pd.to_datetime(m.group(1))
    if m := _MONTH_RE.search(url):
        return pd.to_datetime(f"{m.group(1)}-{m.group(2)}-01")
    return pd.NaT


def analyse_term_frequency(
    jsonl_path: str | Path,
    source_types: list[str] | None = None,
) -> pd.DataFrame:
    """Counts term incidence per document across the RBA corpus.

    Args:
        jsonl_path: Path to the JSONL corpus file.
        source_types: Optional filter list e.g. ['minutes', 'smp']. None = all.

    Returns:
        DataFrame indexed by date with one column per term showing hit counts,
        plus a 'source' and 'url' column for traceability.
    """
    records = []
    with Path(jsonl_path).open("r", encoding="utf-8") as fh:
        for line in fh:
            if not line.strip():
                continue
            doc = json.loads(line)
            if source_types and doc["source_type"] not in source_types:
                continue
            dt = _extract_date(doc["url"])
            if pd.isna(dt):
                continue
            counts = {
                term: len(pattern.findall(doc["text"]))
                for term, pattern in _TERM_PATTERNS.items()
            }
            if not any(counts.values()):
                continue
            records.append({"date": dt, "source": doc["source_type"], "url": doc["url"], **counts})

    df = (
        pd.DataFrame(records)
        .set_index("date")
        .sort_index()
    )
    return df


def summarise(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregates term counts by year for a high-level incidence view.

    Args:
        df: DataFrame as returned by analyse_term_frequency.

    Returns:
        DataFrame of annual term counts.
    """
    term_cols = list(_TERM_PATTERNS.keys())
    return (
        df[term_cols]
        .groupby(df.index.year)
        .sum()
        .rename_axis("year")
    )

In [5]:
df_terms = analyse_term_frequency("rba_corpus.jsonl")
df_summary = summarise(df_terms)
df_summary

,LNG,liquefied natural gas,gas tax,windfall tax,PRRT,petroleum resources rent tax,gas export,energy security,Santos,Chevron,Shell
year,,,,,,,,,,,
2023,1,0,0,0,0,0,0,0,0,0,1


In [29]:
from __future__ import annotations

import asyncio
import json
import logging
import re
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path

import aiohttp
from bs4 import BeautifulSoup
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

logger = logging.getLogger(__name__)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-AU,en;q=0.9",
}
FETCH_CONCURRENCY = 24
REQUEST_TIMEOUT = 25
RETRY_ATTEMPTS = 3
READ_BUFSIZE = 2 ** 18
MIN_RELEVANCE_HITS = 2
MIN_PARAGRAPH_LENGTH = 60
NON_RETRYABLE_STATUSES = frozenset({401, 403, 404, 410, 451})
PROGRESS_INTERVAL = 25

GDELT_DOC_API = "https://api.gdeltproject.org/api/v2/doc/doc"
GDELT_MAX_RECORDS = 250
GDELT_WINDOW_DAYS = 30
GDELT_LOOKBACK_DAYS = 365 * 4

QUERIES: list[str] = [
    '"LNG windfall tax" Australia',
    '"gas windfall tax" Australia',
    '"export tax" LNG Australia',
    '"PRRT" Australia reform',
    '"petroleum resource rent tax" Australia',
    '"gas reservation" Australia',
    '"domestic gas reservation" Australia',
    '"gas trigger" OR ADGSM Australia',
    '"gas price cap" Australia',
    '"gas code of conduct" Australia',
    '"gas market intervention" Australia',
    '"LNG export controls" Australia',
    '"netback" gas Australia',
    'Santos LNG tax Australia',
    'Woodside LNG tax Australia',
    'Chevron Gorgon Australia tax',
    'Shell QGC Australia tax',
    'Inpex Ichthys Australia tax',
    'ExxonMobil Australia gas',
    'Origin Energy gas tax',
    'Beach Energy gas tax',
    'ACCC gas inquiry Australia',
    'AEMO gas shortfall',
    'Gladstone LNG exports',
    '"North West Shelf" gas',
    'Scarborough Pluto LNG',
    'Browse LNG project',
    'Barossa Darwin LNG',
    'Narrabri gas project',
    'Beetaloo basin gas',
    'Albanese gas tax',
    'Chalmers gas tax',
    'Pocock gas tax',
    'Greens gas tax Australia',
    '"Grattan Institute" gas',
    '"Australia Institute" gas tax',
    'IEEFA Australia LNG',
    'Australia LNG Japan METI',
    'Australia LNG Korea',
    'Woodside Santos merger',
    'ADNOC Santos takeover',
    'BlueScope gas Australia',
    'Incitec Pivot gas',
    '"east coast gas" Australia',
    'Australia LNG profits Ukraine',
    'Australia LNG Middle East',
    'Australia gas royalties',
    'Australia fossil fuel subsidies gas',
    'Norway gas tax Australia',
    'Qatar LNG tax Australia',
]

RELEVANCE_PATTERN = re.compile(
    r"\b(LNG|gas tax|windfall|PRRT|petroleum rent|export tax|gas export|"
    r"energy tax|gas price|gas reservation|gas trigger|ADGSM|royalt|"
    r"Santos|Woodside|Chevron|Shell|Inpex|ExxonMobil|Origin Energy|"
    r"Beach Energy|Albanese|Chalmers|Treasury|Pocock|gas market|"
    r"Gladstone|Gorgon|Ichthys|Scarborough|Pluto|Browse|Barossa|"
    r"Narrabri|Beetaloo|North West Shelf|ACCC|AEMO)\b",
    re.IGNORECASE,
)

BLOCKED_TITLES = re.compile(
    r"^(Just In|ABC Sport|ABC Elections|Contact|Environment|Indigenous|"
    r"Tok Pisin|Quizzes|Berita|Find alerts|Sport|Radio|Television|"
    r"iview|Triple J|Subscribe.*|untitled)$",
    re.IGNORECASE,
)


class PermanentFetchError(Exception):
    """Raised on HTTP statuses that should not be retried."""


@dataclass(slots=True)
class NewsDocument:
    url: str
    source: str
    title: str
    text: str
    query: str
    relevance_hits: int


def _gdelt_windows(lookback_days: int, window_days: int) -> list[tuple[str, str]]:
    """Generates contiguous (start, end) timestamp pairs spanning the lookback period.

    GDELT caps each query at 250 records, so partitioning by date window is the
    standard pagination strategy for high-volume topics.

    Args:
        lookback_days: Total days of history to cover.
        window_days: Width of each window in days.

    Returns:
        List of (yyyymmddhhmmss, yyyymmddhhmmss) string tuples.
    """
    fmt = "%Y%m%d%H%M%S"
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=lookback_days)
    windows = []
    cursor = start
    step = timedelta(days=window_days)
    while cursor < end:
        nxt = min(cursor + step, end)
        windows.append((cursor.strftime(fmt), nxt.strftime(fmt)))
        cursor = nxt
    return windows


def _extract_text(html: str) -> str:
    """Extracts substantive paragraph text from an article page.

    Args:
        html: Raw HTML of the article page.

    Returns:
        Newline-separated paragraph text above the minimum length threshold.
    """
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    container = (
        soup.find("article")
        or soup.find("div", class_=re.compile(r"article|content|story|body|post", re.I))
        or soup.find("main")
        or soup.body
    )
    if container is None:
        return ""
    return "\n\n".join(
        p.get_text(separator=" ", strip=True)
        for p in container.find_all("p")
        if len(p.get_text(strip=True)) > MIN_PARAGRAPH_LENGTH
    )


def _extract_title(html: str) -> str:
    """Extracts the page title from an article page.

    Args:
        html: Raw HTML of the article page.

    Returns:
        Title text from h1 or title tag, or 'untitled' if neither exists.
    """
    soup = BeautifulSoup(html, "lxml")
    for tag in ("h1", "title"):
        el = soup.find(tag)
        if el:
            return el.get_text(strip=True)
    return "untitled"


def _make_fetch(session: aiohttp.ClientSession, semaphore: asyncio.Semaphore):
    """Builds a semaphore-bound, retry-decorated async fetch callable.

    Args:
        session: Active aiohttp client session.
        semaphore: Concurrency limiter shared across all fetches.

    Returns:
        Async callable mapping a URL to its response body text.
    """
    @retry(
        retry=retry_if_exception_type((aiohttp.ClientError, asyncio.TimeoutError)),
        stop=stop_after_attempt(RETRY_ATTEMPTS),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        before_sleep=before_sleep_log(logger, logging.DEBUG),
        reraise=True,
    )
    async def _fetch(url: str) -> str:
        async with semaphore:
            async with session.get(url, headers=HEADERS, allow_redirects=True) as r:
                if r.status in NON_RETRYABLE_STATUSES:
                    raise PermanentFetchError(f"{r.status} {url}")
                r.raise_for_status()
                return await r.text(errors="replace")
    return _fetch


async def _gdelt_query(
    session: aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    query: str,
    start: str,
    end: str,
) -> list[tuple[str, str, str]]:
    """Queries one GDELT date window for one search string.

    GDELT returns real publisher URLs as JSON in a single call, so no
    redirect decoding is required.

    Args:
        session: Active aiohttp client session.
        semaphore: Concurrency limiter for HTTP fetches.
        query: GDELT query string (supports phrase and boolean operators).
        start: Window start timestamp (yyyymmddhhmmss).
        end: Window end timestamp (yyyymmddhhmmss).

    Returns:
        List of (article_url, title, source_domain) tuples.
    """
    params = {
        "query": f"{query} sourcecountry:AS",
        "mode": "ArtList",
        "format": "json",
        "maxrecords": str(GDELT_MAX_RECORDS),
        "startdatetime": start,
        "enddatetime": end,
        "sort": "datedesc",
    }
    async with semaphore:
        try:
            async with session.get(GDELT_DOC_API, params=params) as r:
                if r.status != 200:
                    return []
                data = await r.json(content_type=None)
        except Exception:
            return []
    return [
        (a["url"], a.get("title", ""), a.get("domain", ""))
        for a in data.get("articles", [])
        if a.get("url")
    ]


async def _scrape_article(
    fetch, url: str, source: str, query: str, title_hint: str
) -> NewsDocument | None:
    """Fetches and relevance-filters a single news article.

    Returns None on any fetch or parse failure so a single dead URL
    cannot abort the batch.

    Args:
        fetch: Retry-decorated async fetch callable.
        url: Publisher article URL.
        source: Domain label.
        query: GDELT query that surfaced this article.
        title_hint: GDELT title used for pre-fetch rejection.

    Returns:
        Populated NewsDocument if relevance threshold met, else None.
    """
    if BLOCKED_TITLES.match(title_hint):
        return None
    try:
        html = await fetch(url)
    except Exception:
        return None
    title = _extract_title(html) or title_hint
    if BLOCKED_TITLES.match(title):
        return None
    text = _extract_text(html)
    if not text:
        return None
    hits = len(RELEVANCE_PATTERN.findall(f"{title} {text}"))
    if hits < MIN_RELEVANCE_HITS:
        return None
    return NewsDocument(url, source, title, text, query, hits)


async def scrape_news(
    queries: list[str] = QUERIES,
    output_path: str | Path = "news_corpus.jsonl",
    min_relevance_hits: int = MIN_RELEVANCE_HITS,
    lookback_days: int = GDELT_LOOKBACK_DAYS,
    window_days: int = GDELT_WINDOW_DAYS,
) -> int:
    """Scrapes GDELT for all queries and streams relevant articles to JSONL.

    Discovery phase: queries the GDELT DOC API across rolling date windows
    for every search string. Scraping phase: fetches each unique URL through
    a shared concurrency-limited session and writes relevant results as
    they complete.

    Args:
        queries: GDELT search strings.
        output_path: Destination JSONL file path.
        min_relevance_hits: Minimum keyword match count to retain an article.
        lookback_days: Total history depth in days.
        window_days: GDELT pagination window width in days.

    Returns:
        Total number of relevant articles saved.
    """
    fetch_sem = asyncio.Semaphore(FETCH_CONCURRENCY)
    timeout = aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
    connector = aiohttp.TCPConnector(
        limit=FETCH_CONCURRENCY * 2, keepalive_timeout=60, ssl=False
    )
    windows = _gdelt_windows(lookback_days, window_days)
    logger.info(
        "Discovery: %d queries × %d windows = %d GDELT calls",
        len(queries), len(windows), len(queries) * len(windows),
    )
    count = 0

    async with aiohttp.ClientSession(
        connector=connector, timeout=timeout, read_bufsize=READ_BUFSIZE,
        headers=HEADERS,
    ) as session:
        fetch = _make_fetch(session, fetch_sem)

        discovery_tasks = [
            asyncio.create_task(_gdelt_query(session, fetch_sem, q, s, e))
            for q in queries for s, e in windows
        ]
        query_lookup = [
            q for q in queries for _ in windows
        ]

        seen: set[str] = set()
        jobs: list[tuple[str, str, str, str]] = []
        completed = 0
        for task, q in zip(asyncio.as_completed(discovery_tasks), query_lookup):
            results = await task
            completed += 1
            for url, title, domain in results:
                if url in seen:
                    continue
                seen.add(url)
                jobs.append((url, domain or url.split("/")[2], q, title))
            if completed % 20 == 0:
                logger.info("Discovery: %d/%d windows, %d unique URLs",
                            completed, len(discovery_tasks), len(jobs))

        logger.info("Scraping %d unique articles", len(jobs))

        scrape_tasks = [
            asyncio.create_task(_scrape_article(fetch, u, s, q, t))
            for u, s, q, t in jobs
        ]

        with Path(output_path).open("w", encoding="utf-8") as fh:
            attempted = 0
            for coro in asyncio.as_completed(scrape_tasks):
                doc = await coro
                attempted += 1
                if attempted % 100 == 0:
                    logger.info("Progress: %d/%d scraped, %d kept",
                                attempted, len(scrape_tasks), count)
                if doc is None or doc.relevance_hits < min_relevance_hits:
                    continue
                fh.write(json.dumps({
                    "url": doc.url,
                    "source": doc.source,
                    "title": doc.title,
                    "text": doc.text,
                    "query": doc.query,
                    "relevance_hits": doc.relevance_hits,
                }) + "\n")
                count += 1
                if count % PROGRESS_INTERVAL == 0:
                    logger.info("[%d] (%d hits) %s",
                                count, doc.relevance_hits, doc.title[:80])

    return count

In [30]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
total = await scrape_news(output_path="lng_tax_news.jsonl")
print(f"{total} articles saved")

INFO Discovery: 50 queries × 49 windows = 2450 GDELT calls
INFO Discovery: 20/2450 windows, 0 unique URLs
INFO Discovery: 40/2450 windows, 0 unique URLs
INFO Discovery: 60/2450 windows, 0 unique URLs
INFO Discovery: 80/2450 windows, 0 unique URLs
INFO Discovery: 100/2450 windows, 1 unique URLs
INFO Discovery: 120/2450 windows, 5 unique URLs
INFO Discovery: 140/2450 windows, 6 unique URLs
INFO Discovery: 160/2450 windows, 6 unique URLs
INFO Discovery: 180/2450 windows, 6 unique URLs
INFO Discovery: 200/2450 windows, 6 unique URLs
INFO Discovery: 220/2450 windows, 557 unique URLs
INFO Discovery: 240/2450 windows, 1076 unique URLs
INFO Discovery: 260/2450 windows, 1491 unique URLs
INFO Discovery: 280/2450 windows, 1712 unique URLs
INFO Discovery: 300/2450 windows, 1772 unique URLs
INFO Discovery: 320/2450 windows, 1848 unique URLs
INFO Discovery: 340/2450 windows, 1849 unique URLs
INFO Discovery: 360/2450 windows, 1849 unique URLs
INFO Discovery: 380/2450 windows, 1849 unique URLs
INFO Di

1219 articles saved


In [31]:
import json
import pandas as pd

docs = [json.loads(l) for l in open("lng_tax_news.jsonl")]
pd.DataFrame([{"source": d["source"], "title": d["title"], "query": d["query"]} for d in docs])

,source,title,query
0,afr.com,ASX jumps 2.2pc in strongest session in a year...,"""export tax"" LNG Australia"
1,afr.com,Why young Australians are at risk of a poorer ...,"""petroleum resource rent tax"" Australia"
2,oberonreview.com.au,Treasury's intergenerational report paints ble...,"""petroleum resource rent tax"" Australia"
3,stockhead.com.au,GOT GAS: Why the new ‘gas-led recovery’ makes ...,"""gas windfall tax"" Australia"
4,macrobusiness.com.au,War-profiteering energy shocks Australia towar...,"""export tax"" LNG Australia"
...,...,...,...
1214,skynews.com.au,Tamboran Resources says 100 per cent of its pr...,"""domestic gas reservation"" Australia"
1215,macrobusiness.com.au,Only gas can meet the Paris Agreement,"""domestic gas reservation"" Australia"
1216,abnnewswire.net,State Gas Limited (ASX:GAS) Chairman's Address...,"""domestic gas reservation"" Australia"
1217,macrobusiness.com.au,Canberra must intervene radically in failed ga...,"""domestic gas reservation"" Australia"


In [32]:
docs

[{'url': 'https://www.afr.com/markets/equity-markets/asx-to-leap-wall-st-surges-on-hopes-war-will-end-soon-20260401-p5zkgr',
  'source': 'afr.com',
  'title': 'ASX jumps 2.2pc in strongest session in a year; gold stocks soar',
  'text': 'Major banks gain lending share, Macquarie eyes 10pc mortgage hold: UBS\n\nUS-China rivalry to drive earnings risk and policy shocks: CBA\n\nPEXA denies undisclosed information, blames regulator review for 16pc dive\n\nThe Australian sharemarket bounced back on Wednesday in its biggest rise in a year on optimism the Iran war, which has rattled markets and disrupted energy supplies, may be nearing an end.\n\nThe S & P/ASX 200 Index surged 2.2 per cent, or by 190 points, to 8671.80, with ten of the 11 sectors higher.\n\nThe upbeat start – the strongest gain since April 2025 — added $68 billion to the benchmark’s capitalisation. It follows a 7.8 per cent sell-off in March – the steepest monthly drop since June 2022 – driven by fears of a drawn-out conflict

In [36]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from urllib.parse import urlparse


class NewsSentimentBuilder(MacroSentimentBuilder):
    """Builds a daily sentiment index from a news JSONL corpus.

    Uses headline + lead-paragraph scoring rather than full-document
    chunking, which suits short-form news and avoids FinBERT's truncation
    losses on long articles.
    """

    _URL_DATE_RE = re.compile(r"/(\d{4})[/-](\d{1,2})[/-](\d{1,2})")
    _URL_YEARMONTH_RE = re.compile(r"/(\d{4})[/-](\d{1,2})/")
    _LEAD_CHARS = 1500
    _BATCH_SIZE = 32

    def build_index(self, jsonl_path):
        records = self._load_records(Path(jsonl_path))
        df = pd.DataFrame(records).dropna(subset=["date"])
        df = df[df["sentiment"] != 0.0]
        daily = (
            df.groupby("date")
            .agg(sentiment=("sentiment", "mean"), n=("sentiment", "size"))
            .sort_index()
        )
        daily["sentiment_smoothed"] = (
            daily["sentiment"].rolling(window=7, min_periods=1).mean()
        )
        return daily

    def _extract_date(self, url: str) -> pd.Timestamp:
        """Parses a publication date from a news article URL.

        Args:
            url: Article URL, expected to embed a date in its path.

        Returns:
            Parsed Timestamp, or NaT if no date pattern matches.
        """
        path = urlparse(url).path
        if m := self._URL_DATE_RE.search(path):
            try:
                return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))
            except ValueError:
                pass
        if m := self._URL_YEARMONTH_RE.search(path):
            try:
                return pd.Timestamp(int(m.group(1)), int(m.group(2)), 1)
            except ValueError:
                pass
        return pd.NaT

    def _score_batch(self, snippets: list[str]) -> list[float]:
        """Scores a batch of text snippets with one classifier call.

        Args:
            snippets: Pre-truncated text snippets, one per document.

        Returns:
            List of signed sentiment scores aligned with the input order.
        """
        if not snippets:
            return []
        label_sign = {"positive": 1.0, "negative": -1.0, "neutral": 0.0}
        results = self.classifier(snippets, batch_size=self._BATCH_SIZE)
        return [label_sign.get(r["label"], 0.0) * r["score"] for r in results]

    def _load_records(self, path: Path) -> list[dict[str, Any]]:
        """Streams a news JSONL file and batch-scores all dated documents.

        Args:
            path: Path to the news corpus JSONL file.

        Returns:
            List of dicts with 'date', 'source', and 'sentiment' keys.
        """
        rows: list[dict[str, Any]] = []
        snippets: list[str] = []
        with path.open("r", encoding="utf-8") as fh:
            for line in fh:
                if not line.strip():
                    continue
                doc = json.loads(line)
                dt = self._extract_date(doc["url"])
                if pd.isna(dt):
                    continue
                snippet = f"{doc.get('title', '')}. {doc['text'][:self._LEAD_CHARS]}"
                rows.append({"date": dt, "source": doc.get("source", "")})
                snippets.append(snippet)

        scores = self._score_batch(snippets)
        for row, score in zip(rows, scores):
            row["sentiment"] = score
        return rows

In [37]:
df_news = NewsSentimentBuilder(device=-1).build_index("lng_tax_news.jsonl")
df_news.to_csv("lng_news_sentiment.csv")

INFO HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 41961.83it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/main "HTTP/1.1 200 OK"


In [38]:
df_news

,sentiment,n,sentiment_smoothed
date,,,
2022-05-01,-0.618377,1,-0.618377
2022-07-01,-0.294227,3,-0.456302
2022-08-01,-0.415488,3,-0.442697
2022-09-01,-0.929108,2,-0.564300
2022-10-01,-0.753521,5,-0.602144
2022-12-01,-0.657133,7,-0.611309
2022-12-17,-0.615152,1,-0.611858
2023-03-01,-0.852549,1,-0.645311
2023-04-26,-0.501445,1,-0.674914
